## AlexNet Main Base Model -2012

In [2]:
import torch
import torch.nn as nn

In [4]:
class AlexNet(nn.Module):
    def __init__(self, num_classes: int = 1000):
        super(AlexNet, self).__init__()

        self.features=nn.Sequential(
        # Layer 1
        nn.Conv2d(in_channels=3,out_channels=96,kernel_size=11,stride=4,padding=2),
        nn.ReLU(inplace=True),
        nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
        nn.MaxPool2d(kernel_size=3, stride=2),

        # Layer 2
        nn.Conv2d(in_channels=96,out_channels=256,kernel_size=5,stride=1,padding=2),
        nn.ReLU(inplace=True),
        nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
        nn.MaxPool2d(kernel_size=3, stride=2),

        # Layer 3
        nn.Conv2d(in_channels=256,out_channels=384,kernel_size=3,stride=1,padding=2),
        nn.ReLU(inplace=True),
        # nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),

        # Layer 4
        nn.Conv2d(in_channels=384,out_channels=384,kernel_size=3,stride=1,padding=2),
        nn.ReLU(inplace=True),
        # nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),

        # Layer 5
        nn.Conv2d(in_channels=384,out_channels=256,kernel_size=3,stride=1,padding=2),
        nn.ReLU(inplace=True),
        # nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
        nn.MaxPool2d(kernel_size=3, stride=2),

        )

        # Guarantees spatial dimensions of 6x6 before fully connected layers
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        
        self.classifier = nn.Sequential(
            # Layer 6
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            
            # Layer 7
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            
            # Layer 8
            nn.Linear(4096, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.features(x)
            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.classifier(x)
            return x

# Verification script
if __name__ == "__main__":
    model = AlexNet(num_classes=10)
    sample_input = torch.randn(1, 3, 224, 224)
    output = model(sample_input)
    
    print(f"Input batch shape:  {sample_input.shape}")
    print(f"Output batch shape: {output.shape}")
        
        

Input batch shape:  torch.Size([1, 3, 224, 224])
Output batch shape: torch.Size([1, 10])


### Key Innovations in AlexNet

- **ReLU Activation**: While LeNet used tanh activations, AlexNet introduced ReLU, which accelerates convergence and reduces training time.
- **GPU Utilization**: AlexNet was one of the first deep learning models to leverage GPU parallelism, using two GPUs in training to handle the large model and dataset.
- **Dropout Regularization**: AlexNet introduced dropout, a regularization technique that randomly “drops” neurons during training to reduce overfitting.
- **Data Augmentation**: To further reduce overfitting, AlexNet applied techniques like random cropping and horizontal flipping, significantly expanding the effective dataset.

## VGGNet - 16 layers - 2014

VGG-16 Architecture Breakdown
- Stage 1: 2x Conv(64) $\rightarrow$ MaxPool
- Stage 2: 2x Conv(128) $\rightarrow$ MaxPool
- Stage 3: 3x Conv(256) $\rightarrow$ MaxPool
- Stage 4: 3x Conv(512) $\rightarrow$ MaxPool
- Stage 5: 3x Conv(512) $\rightarrow$ MaxPool
- Classifier: FC(4096) $\rightarrow$ ReLU $\rightarrow$ Dropout $\rightarrow$ FC(4096) $\rightarrow$ ReLU $\rightarrow$ Dropout $\rightarrow$ FC(1000) $\rightarrow$ Softmax

In [5]:
# ============================================================
# VGG-16 ARCHITECTURE CONFIGURATION
# ============================================================
#
# Numbers = number of output channels/features
# 'M'     = Max Pooling layer
#
# VGG-16 has 5 blocks:
#
# Block 1: 64, 64, MaxPool
# Block 2: 128, 128, MaxPool
# Block 3: 256, 256, 256, MaxPool
# Block 4: 512, 512, 512, MaxPool
# Block 5: 512, 512, 512, MaxPool
#
# Input image:
# [Batch, 3, 224, 224]
#
# 3 = RGB channels
# 224 x 224 = image height and width

VGG16_CONFIG = [
    64, 64, 'M',
    128, 128, 'M',
    256, 256, 256, 'M',
    512, 512, 512, 'M',
    512, 512, 512, 'M'
]


# ============================================================
# VGG16 MODEL
# ============================================================

class VGG16(nn.Module):

    def __init__(self, num_classes: int = 1000):
        """
        Constructor of the VGG16 model.

        num_classes:
            Number of output classes.

        For ImageNet:
            num_classes = 1000

        For example, if we have cats vs dogs:
            num_classes = 2
        """

        # Initialize the parent nn.Module class
        super(VGG16, self).__init__()


        # ====================================================
        # FEATURE EXTRACTION PART
        # ====================================================
        #
        # This creates all the convolutional + ReLU +
        # MaxPooling layers.
        #
        # The output of this section will be a feature map.
        #
        self.features = self._make_layers(VGG16_CONFIG)


        # ====================================================
        # ADAPTIVE AVERAGE POOLING
        # ====================================================
        #
        # Whatever spatial size comes from the convolutional
        # layers, convert it to:
        #
        #       7 x 7
        #
        # Since our input is 224 x 224 and VGG has 5 pooling
        # layers:
        #
        # 224 -> 112 -> 56 -> 28 -> 14 -> 7
        #
        # Number of channels remains 512.
        #
        # Therefore:
        #
        # [Batch, 512, 7, 7]
        #
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))


        # ====================================================
        # CLASSIFIER
        # ====================================================
        #
        # The convolutional part extracts features.
        #
        # The classifier takes those features and predicts
        # which class the image belongs to.
        #
        self.classifier = nn.Sequential(

            # Input:
            # 512 channels x 7 x 7 feature map
            #
            # 512 * 7 * 7 = 25088
            #
            # Flattened feature vector:
            # [Batch, 25088]
            #
            # Convert 25088 features -> 4096
            nn.Linear(512 * 7 * 7, 4096),

            # Non-linearity
            # Negative values become 0
            nn.ReLU(inplace=True),

            # Randomly deactivate 50% of neurons during training
            # Helps reduce overfitting
            nn.Dropout(p=0.5),


            # 4096 -> 4096
            nn.Linear(4096, 4096),

            # Another ReLU activation
            nn.ReLU(inplace=True),

            # Another dropout layer
            nn.Dropout(p=0.5),


            # Final classification layer
            #
            # 4096 features -> num_classes
            #
            # If num_classes = 10:
            #
            # [Batch, 4096] -> [Batch, 10]
            #
            nn.Linear(4096, num_classes),
        )


    # ============================================================
    # CREATE CONVOLUTIONAL LAYERS
    # ============================================================

    def _make_layers(self, config: list) -> nn.Sequential:

        # Empty list where we will store all layers
        layers = []


        # Our input image has 3 channels:
        #
        # R = 1 channel
        # G = 1 channel
        # B = 1 channel
        #
        # Therefore:
        #
        # input shape = [Batch, 3, Height, Width]
        #
        in_channels = 3


        # Go through every item in VGG16_CONFIG
        for layer in config:


            # ==================================================
            # IF THE CONFIGURATION SAYS 'M'
            # ==================================================
            #
            # 'M' means MaxPooling.
            #
            # MaxPool2d:
            #
            # kernel_size = 2
            # stride      = 2
            #
            # This reduces height and width by half.
            #
            # Example:
            #
            # [224, 224]
            #
            # becomes
            #
            # [112, 112]
            #
            if layer == 'M':

                layers.append(
                    nn.MaxPool2d(
                        kernel_size=2,
                        stride=2
                    )
                )


            # ==================================================
            # OTHERWISE, 'layer' IS A NUMBER
            # ==================================================
            #
            # Example:
            #
            # layer = 64
            #
            # Create:
            #
            # Conv2d(3 -> 64)
            # ReLU
            #
            else:

                layers.extend([

                    # Convolution layer
                    #
                    # in_channels:
                    #     number of channels coming in
                    #
                    # layer:
                    #     number of channels going out
                    #
                    # kernel_size=3:
                    #     3 x 3 convolution filter
                    #
                    # padding=1:
                    #     keeps height and width unchanged
                    #
                    nn.Conv2d(
                        in_channels,
                        layer,
                        kernel_size=3,
                        padding=1
                    ),


                    # Apply ReLU after convolution
                    nn.ReLU(inplace=True)
                ])


                # The output channels of this convolution
                # become the input channels of the next
                # convolution.
                #
                # Example:
                #
                # Conv: 3 -> 64
                #
                # Now:
                # in_channels = 64
                #
                in_channels = layer


        # Convert the Python list into an nn.Sequential object
        #
        # This allows PyTorch to execute the layers one after
        # another automatically.
        #
        return nn.Sequential(*layers)


    # ============================================================
    # FORWARD PASS
    # ============================================================

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # --------------------------------------------------------
        # STEP 1: FEATURE EXTRACTION
        # --------------------------------------------------------
        #
        # Input:
        #
        # [Batch, 3, 224, 224]
        #
        # After all Conv + ReLU + MaxPool layers:
        #
        # [Batch, 512, 7, 7]
        #
        x = self.features(x)


        # --------------------------------------------------------
        # STEP 2: ADAPTIVE AVERAGE POOL
        # --------------------------------------------------------
        #
        # Make sure spatial dimensions are exactly:
        #
        # [7, 7]
        #
        # Output:
        #
        # [Batch, 512, 7, 7]
        #
        x = self.avgpool(x)


        # --------------------------------------------------------
        # STEP 3: FLATTEN
        # --------------------------------------------------------
        #
        # Before:
        #
        # [Batch, 512, 7, 7]
        #
        # 512 * 7 * 7 = 25088
        #
        # After flatten:
        #
        # [Batch, 25088]
        #
        # The "1" means:
        # start flattening from dimension 1.
        #
        # We keep the batch dimension unchanged.
        #
        x = torch.flatten(x, 1)


        # --------------------------------------------------------
        # STEP 4: CLASSIFIER
        # --------------------------------------------------------
        #
        # [Batch, 25088]
        #
        #       ↓ Linear
        #
        # [Batch, 4096]
        #
        #       ↓ Linear
        #
        # [Batch, 4096]
        #
        #       ↓ Linear
        #
        # [Batch, num_classes]
        #
        x = self.classifier(x)


        # Return final prediction
        return x


# ============================================================
# VERIFICATION / TESTING
# ============================================================

if __name__ == "__main__":

    # Create a VGG16 model.
    #
    # Normally VGG16 was designed for 1000 ImageNet classes.
    #
    # Here we use 10 classes just for testing.
    #
    model = VGG16(num_classes=10)


    # Create a fake image for testing.
    #
    # Shape:
    #
    # [1, 3, 224, 224]
    #
    # 1    = batch size
    # 3    = RGB channels
    # 224  = height
    # 224  = width
    #
    # torch.randn generates random numbers from a normal
    # distribution.
    #
    sample_input = torch.randn(1, 3, 224, 224)


    # Send the fake image through the entire VGG16 network.
    #
    # Input:
    # [1, 3, 224, 224]
    #
    # Output:
    # [1, 10]
    #
    output = model(sample_input)


    # Print input shape
    print(f"Input batch shape:  {sample_input.shape}")


    # Print output shape
    print(f"Output batch shape: {output.shape}")

Input batch shape:  torch.Size([1, 3, 224, 224])
Output batch shape: torch.Size([1, 10])


**Key Architectural Innovations**

- Stacked $3 \times 3$ Filters: Replaced large initial filters (like AlexNet's $11 \times 11$) with stacks of small $3 \times 3$ filters. Two stacked $3 \times 3$ layers cover an effective receptive field of $5 \times 5$, while three cover $7 \times 7$.
- Increased Non-Linearity: Using multiple smaller layers instead of one large layer inserts more activation functions ($\text{ReLU}$), allowing the network to learn more complex features.
- Parameter Savings: Stacking three $3 \times 3$ layers uses $3 \times (3^2 \cdot C^2) = 27C^2$ weights, compared to a single $7 \times 7$ layer which requires $1 \times (7^2 \cdot C^2) = 49C^2$ weights—a 45% reduction in parameter count for the same receptive field.
- Systematic Doubling: Feature map channels systematically double after every max-pooling layer ($64 \rightarrow 128 \rightarrow 256 \rightarrow 512$).

## Google Net / Inception Paper 

Inception/GoogLeNet
- Parallel branches
- 1×1 + 3×3 + 5×5 + pooling
- Much more computationally efficient; uses Sparse Matrix
- ~6.8M parameters in GoogLeNet
- More complex architecture
- Uses architectural branching + bottlenecks

In [6]:
class InceptionBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        ch1x1,
        ch3x3_reduce,
        ch3x3,
        ch5x5_reduce,
        ch5x5,
        pool_proj
    ):
        super().__init__()

        # ==========================================
        # BRANCH 1
        # 1×1 convolution
        # ==========================================

        self.branch1 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                ch1x1,
                kernel_size=1
            ),
            nn.ReLU(inplace=True)
        )


        # ==========================================
        # BRANCH 2
        # 1×1 → 3×3
        # ==========================================

        self.branch2 = nn.Sequential(
            # Reduce channels first
            nn.Conv2d(
                in_channels,
                ch3x3_reduce,
                kernel_size=1
            ),
            nn.ReLU(inplace=True),

            # Extract spatial features
            nn.Conv2d(
                ch3x3_reduce,
                ch3x3,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )


        # ==========================================
        # BRANCH 3
        # 1×1 → 5×5
        # ==========================================

        self.branch3 = nn.Sequential(
            # Reduce channels first
            nn.Conv2d(
                in_channels,
                ch5x5_reduce,
                kernel_size=1
            ),
            nn.ReLU(inplace=True),

            # Larger receptive field
            nn.Conv2d(
                ch5x5_reduce,
                ch5x5,
                kernel_size=5,
                padding=2
            ),
            nn.ReLU(inplace=True)
        )


        # ==========================================
        # BRANCH 4
        # 3×3 MaxPool → 1×1
        # ==========================================

        self.branch4 = nn.Sequential(
            nn.MaxPool2d(
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.Conv2d(
                in_channels,
                pool_proj,
                kernel_size=1
            ),
            nn.ReLU(inplace=True)
        )


    def forward(self, x):

        # Run all four branches in parallel

        branch1 = self.branch1(x)

        branch2 = self.branch2(x)

        branch3 = self.branch3(x)

        branch4 = self.branch4(x)


        # Concatenate along CHANNEL dimension
        #
        # x shape = [B, C, H, W]
        #
        # dim=1 means C

        output = torch.cat(
            [branch1, branch2, branch3, branch4],
            dim=1
        )

        return output

#### Create the first part of GoogLeNet

In [8]:
class GoogLeNet(nn.Module):

    def __init__(self, num_classes=1000):

        super().__init__()

        # ==========================================
        # STEM
        # ==========================================

        self.stem = nn.Sequential(

            # Input:
            # [B, 3, 224, 224]
            #
            # Output:
            # [B, 64, 112, 112]

            nn.Conv2d(
                3,
                64,
                kernel_size=7,
                stride=2,
                padding=3
            ),

            nn.ReLU(inplace=True),

            # [B, 64, 112, 112]
            #        ↓
            # [B, 64, 56, 56]

            nn.MaxPool2d(
                kernel_size=3,
                stride=2,
                padding=1
            )
        )
        self.stem2 = nn.Sequential(

            # 1×1 convolution
            #
            # 64 → 64 channels

            nn.Conv2d(
                64,
                64,
                kernel_size=1
            ),

            nn.ReLU(inplace=True),


            # 3×3 convolution
            #
            # 64 → 192 channels

            nn.Conv2d(
                64,
                192,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),


            # Reduce spatial size

            nn.MaxPool2d(
                kernel_size=3,
                stride=2,
                padding=1
            )
        )
        self.inception3a = InceptionBlock(
            in_channels=192,

            ch1x1=64,

            ch3x3_reduce=96,
            ch3x3=128,

            ch5x5_reduce=16,
            ch5x5=32,

            pool_proj=32
        )
        self.inception3b = InceptionBlock(
            in_channels=256,

            ch1x1=128,

            ch3x3_reduce=128,
            ch3x3=192,

            ch5x5_reduce=32,
            ch5x5=96,

            pool_proj=64
        )
        self.pool3 = nn.MaxPool2d(
            kernel_size=3,
            stride=2,
            padding=1
        )
        self.inception4a = InceptionBlock(
            480,
            192,
            96,
            208,
            16,
            48,
            64
        )

        self.inception4b = InceptionBlock(
            512,
            160,
            112,
            224,
            24,
            64,
            64
        )

        self.inception4c = InceptionBlock(
            512,
            128,
            128,
            256,
            24,
            64,
            64
        )

        self.inception4d = InceptionBlock(
            512,
            112,
            144,
            288,
            32,
            64,
            64
        )

        self.inception4e = InceptionBlock(
            528,
            256,
            160,
            320,
            32,
            128,
            128
        )
        self.pool4 = nn.MaxPool2d(
            kernel_size=3,
            stride=2,
            padding=1
        )
        self.inception5a = InceptionBlock(
            832,
            256,
            160,
            320,
            32,
            128,
            128
        )

        self.inception5b = InceptionBlock(
            832,
            384,
            192,
            384,
            48,
            128,
            128
        )
        self.avgpool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )
        self.fc = nn.Linear(
            1024,
            num_classes
        )
    def forward(self, x):

        # ==========================================
        # STEM
        # ==========================================

        x = self.stem(x)

        x = self.stem2(x)


        # ==========================================
        # INCEPTION 3
        # ==========================================

        x = self.inception3a(x)

        x = self.inception3b(x)

        x = self.pool3(x)


        # ==========================================
        # INCEPTION 4
        # ==========================================

        x = self.inception4a(x)

        x = self.inception4b(x)

        x = self.inception4c(x)

        x = self.inception4d(x)

        x = self.inception4e(x)

        x = self.pool4(x)


        # ==========================================
        # INCEPTION 5
        # ==========================================

        x = self.inception5a(x)

        x = self.inception5b(x)


        # ==========================================
        # GLOBAL AVERAGE POOLING
        # ==========================================

        x = self.avgpool(x)


        # ==========================================
        # FLATTEN
        # ==========================================

        x = torch.flatten(x, 1)


        # ==========================================
        # CLASSIFIER
        # ==========================================

        x = self.fc(x)

        return x

In [14]:
if __name__ == "__main__":

    model = GoogLeNet(num_classes=1000)

    x = torch.randn(
        1, 3, 224, 224
    )

    output = model(x)

    print("Input :", x.shape)
    print("Output:", output.shape)

Input : torch.Size([1, 3, 224, 224])
Output: torch.Size([1, 1000])


- If you look closely at the architecture, you will find the number of channels increasing and the size of the image decreasing.
- Image Channel: 3 → 64 → 192 → 256 → 480 → 512 → 832 → 1024
- Image Size: 224 × 224 -> 112 × 112 -> 56 × 56 -> 28 × 28 -> 14 × 14 -> 7 × 7 -> 1 × 1

## SqueezeNet 2016

SqueezeNet is a lightweight CNN architecture introduced in 2016. Its main goal was:

Get AlexNet-level ImageNet accuracy with dramatically fewer parameters.

The original paper reports about 50× fewer parameters than AlexNet, and with model compression the model could be made smaller than 0.5 MB.

|                     | Inception                                   | SqueezeNet                   |
| ------------------- | ------------------------------------------- | ---------------------------- |
| Main building block | Inception module                            | Fire module                  |
| 1×1 conv            | Yes                                         | Yes                          |
| 3×3 conv            | Yes                                         | Yes                          |
| 5×5 conv            | Original Inception                          | No                           |
| Parallel branches   | Multiple                                    | 2 expand branches            |
| Main goal           | Multi-scale feature extraction + efficiency | Extreme parameter efficiency |
| Bottleneck          | 1×1 before expensive conv                   | 1×1 squeeze before expansion |

Main Component of this Architecture

* SqueezeNet is built on top of the Inception research paper
* In the last layer, instead of the last fully connected layer, it uses GAP (Global Average Pooling)
* In Inception, it uses 1 X 1 and 3 X 3 layers and also Squeeze the number of channels to reduce parameters

In [3]:
import torch
import torch.nn as nn


class FireModule(nn.Module):
    """
    Fire Module used in SqueezeNet.

    A Fire Module consists of:
        1. Squeeze layer: 1x1 convolution
        2. Expand layer:
            - 1x1 convolution
            - 3x3 convolution

    The outputs of the two expand branches are concatenated.
    """

    def __init__(
        self,
        in_channels,
        squeeze_channels,
        expand_1x1_channels,
        expand_3x3_channels
    ):
        super().__init__()

        # ---------------------------------------------------------
        # Squeeze layer
        # ---------------------------------------------------------
        # 1x1 convolution reduces the number of channels.
        self.squeeze = nn.Conv2d(
            in_channels=in_channels,
            out_channels=squeeze_channels,
            kernel_size=1
        )

        # ReLU activation after squeeze convolution
        self.squeeze_activation = nn.ReLU(inplace=True)

        # ---------------------------------------------------------
        # Expand layer - 1x1 branch
        # ---------------------------------------------------------
        self.expand_1x1 = nn.Conv2d(
            in_channels=squeeze_channels,
            out_channels=expand_1x1_channels,
            kernel_size=1
        )

        # ---------------------------------------------------------
        # Expand layer - 3x3 branch
        # ---------------------------------------------------------
        # Padding = 1 preserves height and width.
        self.expand_3x3 = nn.Conv2d(
            in_channels=squeeze_channels,
            out_channels=expand_3x3_channels,
            kernel_size=3,
            padding=1
        )

        # ReLU activation after both expand convolutions
        self.expand_activation = nn.ReLU(inplace=True)

    def forward(self, x):

        # ---------------------------------------------------------
        # Step 1: Squeeze
        # ---------------------------------------------------------
        x = self.squeeze(x)
        x = self.squeeze_activation(x)

        # ---------------------------------------------------------
        # Step 2: Expand using 1x1 convolution
        # ---------------------------------------------------------
        expand_1x1 = self.expand_1x1(x)

        # ---------------------------------------------------------
        # Step 3: Expand using 3x3 convolution
        # ---------------------------------------------------------
        expand_3x3 = self.expand_3x3(x)

        # ---------------------------------------------------------
        # Step 4: Concatenate both branches
        #
        # dim=1 means channel dimension in NCHW format:
        # N = batch
        # C = channels
        # H = height
        # W = width
        # ---------------------------------------------------------
        x = torch.cat(
            [expand_1x1, expand_3x3],
            dim=1
        )

        # Apply ReLU after concatenation
        x = self.expand_activation(x)

        return x

In [4]:
class SqueezeNet(nn.Module):
    """
    SqueezeNet 1.0 implementation.

    Default input:
        RGB image -> 3 channels

    Default output:
        1000 classes (ImageNet)
    """

    def __init__(self, num_classes=1000):
        super().__init__()

        # =========================================================
        # Feature Extraction
        # =========================================================

        self.features = nn.Sequential(

            # -----------------------------------------------------
            # Initial convolution
            # -----------------------------------------------------
            # Input:
            #   3 channels
            #
            # Output:
            #   96 channels
            #
            # Kernel:
            #   7x7
            #
            # Stride:
            #   2
            # -----------------------------------------------------
            nn.Conv2d(
                in_channels=3,
                out_channels=96,
                kernel_size=7,
                stride=2
            ),

            nn.ReLU(inplace=True),

            # Downsample spatial dimensions
            nn.MaxPool2d(
                kernel_size=3,
                stride=2,
                ceil_mode=True
            ),

            # -----------------------------------------------------
            # Fire2
            # -----------------------------------------------------
            FireModule(
                in_channels=96,
                squeeze_channels=16,
                expand_1x1_channels=64,
                expand_3x3_channels=64
            ),

            # -----------------------------------------------------
            # Fire3
            # -----------------------------------------------------
            FireModule(
                in_channels=128,
                squeeze_channels=16,
                expand_1x1_channels=64,
                expand_3x3_channels=64
            ),

            # Downsampling
            nn.MaxPool2d(
                kernel_size=3,
                stride=2,
                ceil_mode=True
            ),

            # -----------------------------------------------------
            # Fire4
            # -----------------------------------------------------
            FireModule(
                in_channels=128,
                squeeze_channels=32,
                expand_1x1_channels=128,
                expand_3x3_channels=128
            ),

            # -----------------------------------------------------
            # Fire5
            # -----------------------------------------------------
            FireModule(
                in_channels=256,
                squeeze_channels=32,
                expand_1x1_channels=128,
                expand_3x3_channels=128
            ),

            # Downsampling
            nn.MaxPool2d(
                kernel_size=3,
                stride=2,
                ceil_mode=True
            ),

            # -----------------------------------------------------
            # Fire6
            # -----------------------------------------------------
            FireModule(
                in_channels=256,
                squeeze_channels=48,
                expand_1x1_channels=192,
                expand_3x3_channels=192
            ),

            # -----------------------------------------------------
            # Fire7
            # -----------------------------------------------------
            FireModule(
                in_channels=384,
                squeeze_channels=48,
                expand_1x1_channels=192,
                expand_3x3_channels=192
            ),

            # -----------------------------------------------------
            # Fire8
            # -----------------------------------------------------
            FireModule(
                in_channels=384,
                squeeze_channels=64,
                expand_1x1_channels=256,
                expand_3x3_channels=256
            ),

            # -----------------------------------------------------
            # Fire9
            # -----------------------------------------------------
            FireModule(
                in_channels=512,
                squeeze_channels=64,
                expand_1x1_channels=256,
                expand_3x3_channels=256
            ),
        )

        # =========================================================
        # Classification Layer
        # =========================================================

        self.classifier = nn.Sequential(

            # Dropout helps reduce overfitting
            nn.Dropout(p=0.5),

            # -----------------------------------------------------
            # Final convolution
            #
            # Instead of using a fully connected layer, SqueezeNet
            # uses a 1x1 convolution to produce class scores.
            # -----------------------------------------------------
            nn.Conv2d(
                in_channels=512,
                out_channels=num_classes,
                kernel_size=1
            ),

            nn.ReLU(inplace=True),

            # -----------------------------------------------------
            # Global Average Pooling
            #
            # Converts:
            #
            # [Batch, Classes, H, W]
            #
            # into:
            #
            # [Batch, Classes, 1, 1]
            # -----------------------------------------------------
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):

        # Extract visual features
        x = self.features(x)

        # Generate class scores
        x = self.classifier(x)

        # Remove unnecessary dimensions
        #
        # [Batch, Classes, 1, 1]
        #          ↓
        # [Batch, Classes]
        x = torch.flatten(x, 1)

        return x

In [5]:
import torch

# ---------------------------------------------------------
# Create model
# ---------------------------------------------------------

model = SqueezeNet(num_classes=1000)

# Put model into evaluation mode
model.eval()

# ---------------------------------------------------------
# Create a dummy input
#
# Batch size = 1
# Channels = 3 (RGB)
# Image size = 224 x 224
# ---------------------------------------------------------

x = torch.randn(
    1,
    3,
    224,
    224
)

# ---------------------------------------------------------
# Forward pass
# ---------------------------------------------------------

with torch.no_grad():
    output = model(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 1000])


In [7]:
def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


model = SqueezeNet(num_classes=1000)

total_parameters = count_parameters(model)

print(f"Total parameters: {total_parameters:,}")

NameError: name 'SqueezeNet' is not defined

In [7]:
parameter_memory_mb = (
    total_parameters * 4
) / (1024 ** 2)

print(
    f"Approx parameter memory: "
    f"{parameter_memory_mb:.2f} MB"
)

Approx parameter memory: 4.76 MB


## ResNet 2015

Before ResNet, people thought:

VGG-16
   ↓
VGG-19
   ↓
Make it deeper
   ↓
50 layers
   ↓
100 layers

But there was a problem.

You might expect:

More layers → more learning → better accuracy.

But after a certain depth, simply adding layers can make optimization harder. The training error can actually get worse.

This is called the degradation problem.

The ResNet paper was designed to address this by introducing residual learning.

Main Component
* Residual/Skip Connection
* BasicBlock
* Batch Normalization
* Global Average Pooling

In [5]:
import torch
import torch.nn as nn


class BasicBlock(nn.Module):

    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):

        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = None

        if stride != 1 or in_channels != out_channels:

            self.downsample = nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),

                nn.BatchNorm2d(out_channels)
            )


    def forward(self, x):

        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity

        out = self.relu(out)

        return out


class ResNet(nn.Module):

    def __init__(self, num_classes=1000):

        super().__init__()

        self.in_channels = 64

        # Stem
        self.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(64)

        self.relu = nn.ReLU(inplace=True)

        self.maxpool = nn.MaxPool2d(
            kernel_size=3,
            stride=2,
            padding=1
        )

        # ResNet layers
        self.layer1 = self.make_layer(
            64,
            2,
            stride=1
        )

        self.layer2 = self.make_layer(
            128,
            2,
            stride=2
        )

        self.layer3 = self.make_layer(
            256,
            2,
            stride=2
        )

        self.layer4 = self.make_layer(
            512,
            2,
            stride=2
        )

        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.fc = nn.Linear(
            512,
            num_classes
        )


    def make_layer(
        self,
        out_channels,
        blocks,
        stride
    ):

        layers = []

        # First block
        layers.append(
            BasicBlock(
                self.in_channels,
                out_channels,
                stride
            )
        )

        self.in_channels = out_channels

        # Remaining blocks
        for _ in range(1, blocks):

            layers.append(
                BasicBlock(
                    self.in_channels,
                    out_channels
                )
            )

        return nn.Sequential(*layers)


    def forward(self, x):

        # Stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Residual stages
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Global average pooling
        x = self.avgpool(x)

        # Flatten
        x = torch.flatten(x, 1)

        # Classification
        x = self.fc(x)

        return x

In [6]:
model = ResNet(num_classes=10)

x = torch.randn(1, 3, 224, 224)

output = model(x)

print("Input :", x.shape)
print("Output:", output.shape)

Input : torch.Size([1, 3, 224, 224])
Output: torch.Size([1, 10])


In [11]:
def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
model = ResNet(num_classes=1000)
count_para=count_parameters(model)


In [12]:
parameter_memory_mb = (
    count_para * 4
) / (1024 ** 2)

print(
    f"Approx parameter memory: "
    f"{parameter_memory_mb:.2f} MB"
)

Approx parameter memory: 44.59 MB


VGG
│
├── Simple sequential architecture
├── Lots of 3×3 convolutions
└── Huge parameter count
    
        
        ↓
Inception
│
├── Parallel branches
├── 1×1 bottlenecks
└── Multi-scale feature extraction
        ↓

        
SqueezeNet
│
├── Aggressive 1×1 usage
├── Fire modules
└── Very small parameter count
        ↓

        
ResNet
│
├── Skip connections
├── Residual learning
└── Enables very deep networks

## MobileNet 2017

MobileNet reduces computation by replacing a standard convolution with Depthwise Separable Convolution.

Why was MobileNet created?

A normal CNN can be computationally expensive.

For example, suppose we have:

Input:
[B, 32, 112, 112]

and want:

Output:
[B, 64, 112, 112]

Using a normal:

3 × 3 Conv
32 → 64

requires:

$$ 3\times3\times32\times64 $$ $$ =18,432 $$

parameters.

But more importantly, there are many convolution operations.

MobileNet asks:

Can we perform this operation much more cheaply?

The answer is:

Depthwise Separable Convolution 🚀

- Depthwise Convolution
- Pointwise Convolution

In [1]:
import torch
import torch.nn as nn


class DepthwiseSeparableConv(nn.Module):

    def __init__(self, in_channels, out_channels, stride):

        super().__init__()

        self.depthwise = nn.Sequential(

            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_channels,
                bias=False
            ),

            nn.BatchNorm2d(in_channels),
            nn.ReLU6(inplace=True)
        )

        self.pointwise = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU6(inplace=True)
        )


    def forward(self, x):

        x = self.depthwise(x)

        x = self.pointwise(x)

        return x


class MobileNetV1(nn.Module):

    def __init__(self, num_classes=1000):

        super().__init__()

        self.features = nn.Sequential(

            # Initial convolution
            nn.Conv2d(
                3,
                32,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),

            # Depthwise separable blocks
            DepthwiseSeparableConv(32, 64, 1),

            DepthwiseSeparableConv(64, 128, 2),

            DepthwiseSeparableConv(128, 128, 1),

            DepthwiseSeparableConv(128, 256, 2),

            DepthwiseSeparableConv(256, 256, 1),

            DepthwiseSeparableConv(256, 512, 2),

            DepthwiseSeparableConv(512, 512, 1),
            DepthwiseSeparableConv(512, 512, 1),
            DepthwiseSeparableConv(512, 512, 1),
            DepthwiseSeparableConv(512, 512, 1),
            DepthwiseSeparableConv(512, 512, 1),

            DepthwiseSeparableConv(512, 1024, 2),

            DepthwiseSeparableConv(1024, 1024, 1)
        )


        self.avgpool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        self.classifier = nn.Linear(
            1024,
            num_classes
        )


    def forward(self, x):

        x = self.features(x)

        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x

In [2]:
model = MobileNetV1(num_classes=10)

x = torch.randn(1, 3, 224, 224)

output = model(x)

print(x.shape)
print(output.shape)

torch.Size([1, 3, 224, 224])
torch.Size([1, 10])


## DenseNet (2017) and EfficientNet (2019)

- Better Scaling (EfficientNet)
- Better Connectivity (DenseNet)

**DenseNet**

- DenseNet differs from traditional architectures by connecting each layer to all preceding layers within a dense block, rather than just passing the output of the previous layer to the next.
- This approach eliminates the need for layers to learn the same type of features repeatedly.
- It selectively utilizes features from previous layers.
- Overall, this design reduces computational complexity.

**Why was DenseNet created?**

- Problem 1 — Vanishing gradients
- Problem 2 — Feature reuse
- Problem 3 — Parameter efficiency

##### **Dense Layer**

In [1]:
import torch
import torch.nn as nn


class DenseLayer(nn.Module):
    """
    A single DenseNet layer.

    Structure:

        BatchNorm
            ↓
          ReLU
            ↓
        1×1 Conv
            ↓
        BatchNorm
            ↓
          ReLU
            ↓
        3×3 Conv

    The newly generated features are concatenated
    with the original input.
    """

    def __init__(self, in_channels, growth_rate):
        super().__init__()

        # Bottleneck produces 4 × growth_rate channels
        bottleneck_channels = 4 * growth_rate

        # -------------------------------------------------------
        # First Batch Normalization
        # -------------------------------------------------------
        self.bn1 = nn.BatchNorm2d(in_channels)

        # -------------------------------------------------------
        # First ReLU
        # -------------------------------------------------------
        self.relu1 = nn.ReLU(inplace=True)

        # -------------------------------------------------------
        # 1×1 Bottleneck Convolution
        # -------------------------------------------------------
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=bottleneck_channels,
            kernel_size=1,
            bias=False
        )

        # -------------------------------------------------------
        # Second Batch Normalization
        # -------------------------------------------------------
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)

        # -------------------------------------------------------
        # Second ReLU
        # -------------------------------------------------------
        self.relu2 = nn.ReLU(inplace=True)

        # -------------------------------------------------------
        # 3×3 Convolution
        #
        # Padding = 1 keeps H and W unchanged.
        # -------------------------------------------------------
        self.conv2 = nn.Conv2d(
            in_channels=bottleneck_channels,
            out_channels=growth_rate,
            kernel_size=3,
            padding=1,
            bias=False
        )

    def forward(self, x):

        # BN → ReLU → 1×1 Conv
        out = self.conv1(
            self.relu1(
                self.bn1(x)
            )
        )

        # BN → ReLU → 3×3 Conv
        out = self.conv2(
            self.relu2(
                self.bn2(out)
            )
        )

        # -------------------------------------------------------
        # Dense connectivity
        #
        # Original input + newly generated features
        #
        # Concatenation happens along channel dimension.
        # -------------------------------------------------------
        return torch.cat([x, out], dim=1)

#### **DenseBlock**

In [9]:
class DenseBlock(nn.Module):
    """
    A Dense Block consists of multiple DenseLayers.

    Each layer receives the feature maps from ALL
    previous layers.
    """

    def __init__(
        self,
        num_layers,
        in_channels,
        growth_rate
    ):
        super().__init__()

        layers = []

        current_channels = in_channels

        for _ in range(num_layers):

            # Create one DenseLayer
            layers.append(
                DenseLayer(
                    in_channels=current_channels,
                    growth_rate=growth_rate
                )
            )

            # Every DenseLayer adds growth_rate channels
            current_channels += growth_rate

        # Store all DenseLayers
        self.layers = nn.Sequential(*layers)

    def forward(self, x):

        return self.layers(x)

#### Transaction Layer

In [4]:
class TransitionLayer(nn.Module):
    """
    Transition layer between two Dense Blocks.

    Structure:

        BatchNorm
            ↓
          ReLU
            ↓
        1×1 Conv
            ↓
        2×2 Average Pool

    The transition layer:
        1. Reduces the number of channels.
        2. Reduces spatial resolution.
    """

    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        # Batch Normalization
        self.bn = nn.BatchNorm2d(in_channels)

        # ReLU activation
        self.relu = nn.ReLU(inplace=True)

        # 1×1 convolution reduces channels
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            bias=False
        )

        # Average pooling reduces H and W by approximately 2×
        self.pool = nn.AvgPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):

        x = self.bn(x)
        x = self.relu(x)
        x = self.conv(x)
        x = self.pool(x)

        return x

##### **DenseNet**

In [10]:
class DenseNet(nn.Module):
    """
    DenseNet implementation.

    Default configuration corresponds to DenseNet-121:

        Dense blocks:
            6
            12
            24
            16

        Growth rate:
            32

        Initial channels:
            64

        Compression:
            0.5
    """

    def __init__(
        self,
        num_classes=1000,
        growth_rate=32,
        block_config=(6, 12, 24, 16),
        num_init_features=64,
        compression=0.5
    ):
        super().__init__()

        # =======================================================
        # Initial Feature Extraction
        # =======================================================

        self.features = nn.Sequential(

            # Initial 7×7 convolution
            nn.Conv2d(
                in_channels=3,
                out_channels=num_init_features,
                kernel_size=7,
                stride=2,
                padding=3,
                bias=False
            ),

            # Batch Normalization
            nn.BatchNorm2d(num_init_features),

            # ReLU activation
            nn.ReLU(inplace=True),

            # Initial downsampling
            nn.MaxPool2d(
                kernel_size=3,
                stride=2,
                padding=1
            )
        )

        # Track current number of channels
        num_features = num_init_features

        # =======================================================
        # Dense Blocks + Transition Layers
        # =======================================================

        for block_index, num_layers in enumerate(block_config):

            # ---------------------------------------------------
            # Dense Block
            # ---------------------------------------------------

            dense_block = DenseBlock(
                num_layers=num_layers,
                in_channels=num_features,
                growth_rate=growth_rate
            )

            self.features.add_module(
                f"dense_block_{block_index + 1}",
                dense_block
            )

            # Every layer adds growth_rate channels
            num_features += (
                num_layers * growth_rate
            )

            # ---------------------------------------------------
            # Transition Layer
            #
            # Do not add transition after final Dense Block.
            # ---------------------------------------------------

            if block_index != len(block_config) - 1:

                # Compression reduces channels
                out_features = int(
                    num_features * compression
                )

                transition = TransitionLayer(
                    in_channels=num_features,
                    out_channels=out_features
                )

                self.features.add_module(
                    f"transition_{block_index + 1}",
                    transition
                )

                num_features = out_features

        # =======================================================
        # Final BatchNorm
        # =======================================================

        self.features.add_module(
            "final_bn",
            nn.BatchNorm2d(num_features)
        )

        # =======================================================
        # Classification Layer
        # =======================================================

        self.classifier = nn.Linear(
            num_features,
            num_classes
        )

    def forward(self, x):

        # Feature extraction
        x = self.features(x)

        # Global Average Pooling
        #
        # [B, C, H, W]
        #      ↓
        # [B, C, 1, 1]
        #
        x = torch.nn.functional.adaptive_avg_pool2d(
            x,
            (1, 1)
        )

        # Flatten
        #
        # [B, C, 1, 1]
        #      ↓
        # [B, C]
        #
        x = torch.flatten(x, 1)

        # Classification
        x = self.classifier(x)

        return x

In [11]:
# Create DenseNet-121
model = DenseNet(
    num_classes=1000,
    growth_rate=32,
    block_config=(6, 12, 24, 16),
    num_init_features=64,
    compression=0.5
)


# Dummy input
x = torch.randn(
    1,      # batch size
    3,      # RGB channels
    224,    # height
    224     # width
)

# Forward pass
with torch.no_grad():
    output = model(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 1000])


In [12]:
def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


total_params = count_parameters(model)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

Total parameters: 7,978,856


## Efficient Net

EfficientNet scales the network's depth, width, and input resolution together using a principled compound-scaling method.

EfficientNet's Main Idea

EfficientNet says:

Don't arbitrarily increase only depth, width, or resolution. Scale all three in a balanced way.

These three dimensions are:

                 EfficientNet
                      │
       ┌──────────────┼──────────────┐
       ▼              ▼              ▼
     Depth          Width       Resolution
    layers        channels     Image size

EfficientNet does something conceptually like:

Depth       ↑
Width       ↑
Resolution  ↑

in a coordinated manner.

The scaling equations are:

$$ depth = \alpha^\phi $$ $$ width = \beta^\phi $$ $$ resolution = \gamma^\phi $$

where:

\(\phi\) = scaling coefficient
\(\alpha\) = depth coefficient
\(\beta\) = width coefficient
\(\gamma\) = resolution coefficient

subject to approximately:

$$ \alpha \cdot \beta^2 \cdot \gamma^2 \approx 2 $$

This means that when the model gets larger, the available computational budget is approximately doubled per increment of \(\phi\).

- Depth

More layers allow the model to learn increasingly complex representations.

- Width

More channels allow the network to represent more features at each layer.

- Resolution

Higher resolution preserves more spatial information.